# Exp 2 — Experimental Data Pipeline

**Source:** `exp_data/00_raw.xlsx` (78960 rows × 79 cols, experimental trial rows, correct participant IDs)

Each checkpoint is saved as a numbered xlsx file so you can open any step in Excel and go back if needed.

| Step | File | What happened |
|------|------|---------------|
| 00 | `00_raw.xlsx` | Experimental rows from allData_exp2 (no question rows) |
| 01 | `01_filtered.xlsx` | One row per trial, selected columns, acc + rt added |
| 02 | `02_excl_acc.xlsx` | Excluded 4 subjects below 85% accuracy threshold |
| 03 | `03_excl_likert.xlsx` | Excluded 7 careless Likert responders (aligned with questions pipeline) |
| 04 | `04_excl_errors.xlsx` | Error trials (acc = 0) removed |
| 05 | `05_gaze_congruency.xlsx` | gaze_congruency, action_consistency, and initiation_time columns added |
| 06 | `06_trimmed.xlsx` | RT outlier trimming — **Python ±2SD**, per subject × condition (SPSS's `2w.SPS` script has a confirmed bug, see `SPSS_2w_script_bug_summary.txt` in the project root — it under-trims by silently keeping RT=0 trials that its own filter logic flags for exclusion) |
| 07 | `07_no_outliers.xlsx` | Participant-level outlier removed (mean RT > grand mean + 2 SD) |
| 08 | `08_ranova_input.xlsx` | Mean RT per subject × condition — long + wide, ready for rANOVA |
| 09 | `09_initiation_time.xlsx` | Mean initiation time per subject, action trials only (ANCOVA covariate) |
| 10 | `10_questions_data_added.xlsx` | Wide RT data + question ratings + initiation-time covariate merged |
| 11 | `11_merged_initTime_for_ANCOVA.xlsx` | Stable xlsx copy of the merged data (same content as Step 10) — kept in `exp_data/`, won't change if the pipeline re-runs |
| — | `jasp/analysis.csv` | Same data as CSV — open in JASP to analyse (canonical, overwritten every run) |

In [1]:
import os
import pandas as pd
import numpy as np

base = r'C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data'

## Step 01 — Filter to one row per trial, select columns, add acc/rt

**Filters applied:**
1. `Procedure[SubTrial]` = `'pro'` → keeps only experimental sub-trial rows
2. `Running[Trial]` = `'Exp'` → excludes practice trials (PracticeL)
3. `pic` = `'6L'` or `'6R'` → keeps only the last slide per trial (where the response was captured)

**acc/rt logic:** `Running[SubTrial]` says which side was last shown (ListLeft or ListRight). The corresponding `ColorSlideLeft` or `ColorSlideRight` columns hold the actual response data.

In [2]:
df = pd.read_excel(f'{base}\\00_raw.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

Loaded: 78960 rows × 79 cols


In [3]:
# Filter 1: keep only experimental sub-trial rows
df = df[df['Procedure[SubTrial]'] == 'pro'].copy()
print(f'After Procedure[SubTrial]=pro: {len(df)} rows')

# Filter 2: exclude practice trials
df = df[df['Running[Trial]'] == 'Exp'].copy()
print(f'After Running[Trial]=Exp:      {len(df)} rows')

# Filter 3: keep only last-slide rows (where response was captured)
df = df[df['pic'].isin(['6L', '6R'])].copy()
print(f'After pic in [6L, 6R]:         {len(df)} rows')
print(f'Expected:                       {60 * 168} rows (60 subjects × 168 trials)')

After Procedure[SubTrial]=pro: 78960 rows
After Running[Trial]=Exp:      70560 rows
After pic in [6L, 6R]:         10080 rows
Expected:                       10080 rows (60 subjects × 168 trials)


In [4]:
# Verify: each subject should have exactly 168 rows
trial_counts = df.groupby('Subject').size()
bad = trial_counts[trial_counts != 168]
if bad.empty:
    print('All subjects have exactly 168 trials.')
else:
    print('WARNING — unexpected trial counts:')
    print(bad)

All subjects have exactly 168 trials.


In [5]:
# Select columns (Procedure[SubTrial] dropped after filtering)
keep = [
    'Subject', 'Handedness', 'blue', 'green',
    'Choice[Trial]', 'colorLeft', 'colorRight',
    'ColorSlideLeft.ACC', 'ColorSlideLeft.RESP', 'ColorSlideLeft.RT',
    'ColorSlideRight.ACC', 'ColorSlideRight.RESP', 'ColorSlideRight.RT',
    'corR', 'Exp.Sample', 'GazeColorConsistency',
    'Procedure[Trial]', 'SlideXeroA.RT', 'XeroDur',
    'pic', 'Running[SubTrial]'
]
df = df[keep].copy()
print(f'Columns selected: {df.shape[1]}')
df.head(3)

Columns selected: 21


,Subject,Handedness,blue,green,Choice[Trial],colorLeft,colorRight,ColorSlideLeft.ACC,ColorSlideLeft.RESP,ColorSlideLeft.RT,...,ColorSlideRight.RESP,ColorSlideRight.RT,corR,Exp.Sample,GazeColorConsistency,Procedure[Trial],SlideXeroA.RT,XeroDur,pic,Running[SubTrial]
146,1,right,a,d,Left,blue,white,0.0,NaN,0.0,...,NaN,0.0,a,1.0,offline,actionNC,489.0,NaN,6R,ListRight
153,1,right,a,d,Right,white,blue,1.0,NaN,0.0,...,NaN,0.0,a,2.0,offline,actionNC,719.0,NaN,6L,ListLeft
160,1,right,a,d,Left,white,green,1.0,d,440.0,...,NaN,0.0,d,3.0,offline,actionC,584.0,NaN,6L,ListLeft


In [6]:
# Calculate merged acc and rt based on which side recorded the response
left  = df['Running[SubTrial]'] == 'ListLeft'
right = df['Running[SubTrial]'] == 'ListRight'

df['acc'] = np.where(left, df['ColorSlideLeft.ACC'],
            np.where(right, df['ColorSlideRight.ACC'], np.nan))

df['rt']  = np.where(left, df['ColorSlideLeft.RT'],
            np.where(right, df['ColorSlideRight.RT'], np.nan))

print(f'acc nulls: {df["acc"].isna().sum()}  |  rt nulls: {df["rt"].isna().sum()}')
print(f'acc values: {df["acc"].value_counts().to_dict()}')

acc nulls: 0  |  rt nulls: 0
acc values: {1.0: 9491, 0.0: 589}


In [7]:
# ACC per subject
acc_per_sub = df.groupby('Subject')['acc'].mean().mul(100).round(2).reset_index()
acc_per_sub.columns = ['Subject', 'ACC_%']
acc_per_sub['flag'] = acc_per_sub['ACC_%'].apply(
    lambda x: '*** <80%' if x < 80 else ('* <85%' if x < 85 else '')
)
print(acc_per_sub.to_string(index=False))

 Subject  ACC_%     flag
       1  50.00 *** <80%
       2  70.83 *** <80%
       3 100.00         
       4  94.64         
       5  99.40         
       6  96.43         
       7  97.02         
       8  98.81         
       9  95.24         
      10  98.81         
      11  98.81         
      12  96.43         
      13  96.43         
      14  99.40         
      15  94.64         
      16  99.40         
      17  95.83         
      18  95.83         
      19  95.83         
      20  97.02         
      21  92.86         
      22  87.50         
      23  98.21         
      24  98.21         
      25  97.62         
      26  96.43         
      27  93.45         
      28 100.00         
      29  98.21         
      30  97.02         
      40  93.45         
      41  99.40         
      42  98.21         
      43  94.64         
      44  95.24         
      45  66.07 *** <80%
      46  96.43         
      47  98.21         
      48  98.81         


In [8]:
# Save checkpoint 01
out = f'{base}\\01_filtered.xlsx'
df.to_excel(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {df.shape}')

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\01_filtered.xlsx
Shape: (10080, 23)


## Step 02 — Exclude participants below 85% accuracy

Threshold: **85%** (same as Experiment 1).

Questions pipeline doesn't need to know about this exclusion — low-accuracy subjects naturally
drop out later when Step 10's merge keeps only subjects present in this (exp_data's) final sample.

In [9]:
df = pd.read_excel(f'{base}\\01_filtered.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

# ACC per subject
acc_per_sub = df.groupby('Subject')['acc'].mean().mul(100).round(2)

# Identify subjects below threshold
THRESHOLD = 85.0
exclude = acc_per_sub[acc_per_sub < THRESHOLD]
keep_subs = acc_per_sub[acc_per_sub >= THRESHOLD].index

print(f'\nThreshold: {THRESHOLD}%')
print(f'\nExcluded subjects ({len(exclude)}):')
for sub, acc in exclude.sort_values().items():
    print(f'  ss {sub}: {acc:.2f}%')

print(f'\nRemaining: {len(keep_subs)} subjects')

Loaded: 10080 rows × 23 cols

Threshold: 85.0%

Excluded subjects (4):
  ss 1: 50.00%
  ss 45: 66.07%
  ss 2: 70.83%
  ss 59: 81.55%

Remaining: 56 subjects


In [10]:
df2 = df[df['Subject'].isin(keep_subs)].copy()

n_trials = len(df2)
n_subjects = df2['Subject'].nunique()
print(f'Remaining after exclusion:')
print(f'  Subjects: {n_subjects}')
print(f'  Trials:   {n_trials}  ({n_subjects} × 168 = {n_subjects * 168})')

out = f'{base}\\02_excl_acc.xlsx'
df2.to_excel(out, index=False)
print(f'\nSaved: {out}')

Remaining after exclusion:
  Subjects: 56
  Trials:   9408  (56 × 168 = 9408)

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\02_excl_acc.xlsx


## Step 03 — Exclude careless Likert responders

Reads the excluded-subject list `questions/pipeline.ipynb` Step 03 writes directly
(`03_likert_excluded_subjects.txt`) — no recomputation from the report, no hardcoded list.

**Excluded (7):** ss 9, 15, 22, 42, 49, 65, 66 — >75% identical Likert responses, near-zero range across conditions  
**Kept despite flag:** ss 41 (range = 1.98), ss 54 (range = 2.00) — discriminate between conditions

In [11]:
with open(os.path.join(os.path.dirname(base), 'questions', '03_likert_excluded_subjects.txt')) as f:
    content = f.read().strip()
    LIKERT_EXCLUDED = [int(x) for x in content.split(',') if x] if content else []
print(f'Read from questions pipeline: {LIKERT_EXCLUDED}')

df = pd.read_excel(f'{base}\\02_excl_acc.xlsx')
df3 = df[~df['Subject'].isin(LIKERT_EXCLUDED)].copy()

n_subjects = df3['Subject'].nunique()
n_trials = len(df3)
print(f'Remaining subjects: {n_subjects}')
print(f'Remaining trials:   {n_trials}  ({n_subjects} x 168 = {n_subjects * 168})')

out = f'{base}\\03_excl_likert.xlsx'
df3.to_excel(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {df3.shape}')

Read from questions pipeline: [9, 15, 22, 42, 49, 65, 66]
Remaining subjects: 49
Remaining trials:   8232  (49 x 168 = 8232)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\03_excl_likert.xlsx
Shape: (8232, 23)


## Step 04 — Remove error trials

Error trials: `acc = 0` — participant pressed the wrong key or timed out.

Removed before RT analysis. The 85% threshold in Step 02 ensures all remaining subjects are high-performers overall; this step removes the residual incorrect trials within those subjects.

In [12]:
df = pd.read_excel(f'{base}\\03_excl_likert.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

n_before = len(df)
n_errors = (df['acc'] == 0).sum()
print(f'\nError trials (acc = 0): {n_errors}  ({n_errors / n_before * 100:.2f}%)')

# Errors per subject (only subjects with at least one error)
errors_per_sub = df[df['acc'] == 0].groupby('Subject').size()
print(f'\nErrors per subject (non-zero only):')
print(errors_per_sub.to_string())

df4 = df[df['acc'] == 1].copy()

n_subjects = df4['Subject'].nunique()
n_trials = len(df4)
print(f'\nRemaining after error removal:')
print(f'  Subjects: {n_subjects}')
print(f'  Trials:   {n_trials}  (removed {n_before - n_trials} error trials)')

out = f'{base}\\04_excl_errors.xlsx'
df4.to_excel(out, index=False)
print(f'\nSaved: {out}')
print(f'Shape: {df4.shape}')

Loaded: 8232 rows × 23 cols

Error trials (acc = 0): 319  (3.88%)

Errors per subject (non-zero only):
Subject
4      9
5      1
6      6
7      5
8      2
10     2
11     2
12     6
13     6
14     1
16     1
17     7
18     7
19     7
20     5
21    12
23     3
24     3
25     4
26     6
27    11
29     3
30     5
40    11
41     1
43     9
44     8
46     6
47     3
48     2
50    14
51    20
52     5
54     9
55     2
56    13
57    10
58     4
60     7
61    14
62     4
63    14
64     3
67    22
68     9
69    10
70     5

Remaining after error removal:
  Subjects: 49
  Trials:   7913  (removed 319 error trials)

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\04_excl_errors.xlsx
Shape: (7913, 23)


## Step 05 — Calculate gaze_congruency and action_consistency

Logic follows Experiment 1 (`raw+processed.xlsx`, tab `cleaned`):

| Column | Source | Logic |
|--------|--------|-------|
| `actual_color_location` | `colorLeft` | `colorLeft == "white"` → "right", else "left" |
| `actual_gaze_dir` | `Running[SubTrial]` | `ListLeft` → "left", `ListRight` → "right" |
| `wanted_gaze_dir` | `Choice[Trial]` / `Procedure[Trial]` | action: `Choice[Trial].lower()`; control: from condition name |
| `gaze_congruency` | derived | `actual_color_location == actual_gaze_dir` → "congruent", else "incongruent" |
| `action_consistency` | `Procedure[Trial]` | actionC → "consistent", actionNC → "inconsistent", control → "control" |

`gaze_congruency` is calculated from raw columns — **not** from E-Prime's `GazeColorConsistency` (which has an "offline" category and a typo). `action_consistency` is a direct mapping — simpler than Exp 1's formula.

In [14]:
df = pd.read_excel(f'{base}\\04_excl_errors.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

# actual_color_location: which side did the target color appear on?
# colorLeft = "white" means no color on left → color is on the right
df['actual_color_location'] = np.where(df['colorLeft'] == 'white', 'right', 'left')

# actual_gaze_dir: which direction did the gaze end up?
df['actual_gaze_dir'] = np.where(df['Running[SubTrial]'] == 'ListLeft', 'left', 'right')

# wanted_gaze_dir: intended direction (action: from Choice[Trial]; control: from condition name)
df['wanted_gaze_dir'] = np.where(
    df['Choice[Trial]'].isin(['Left', 'Right']),
    df['Choice[Trial]'].str.lower(),
    np.where(df['Procedure[Trial]'] == 'controlLeft',  'left',
    np.where(df['Procedure[Trial]'] == 'controlRight', 'right', ''))
)

# gaze_congruency
df['gaze_congruency'] = np.where(
    df['actual_color_location'] == df['actual_gaze_dir'], 'congruent', 'incongruent'
)

# action_consistency: direct mapping from Procedure[Trial]
df['action_consistency'] = df['Procedure[Trial]'].map({
    'actionC':      'consistent',
    'actionNC':     'inconsistent',
    'controlLeft':  'control',
    'controlRight': 'control',
})

# initiation_time: merge SlideXeroA.RT (action trials) and XeroDur (control trials)
# SlideXeroA.RT is NaN for control trials; XeroDur is NaN for action trials
# fillna picks the correct value for each condition automatically
df['initiation_time'] = df['SlideXeroA.RT'].fillna(df['XeroDur'])

# Sanity checks
print('\ngaze_congruency:')
print(df['gaze_congruency'].value_counts())
print('\naction_consistency:')
print(df['action_consistency'].value_counts())

# Cross-check: compare calculated gaze_congruency vs E-Prime GazeColorConsistency
print('\nE-Prime GazeColorConsistency vs calculated gaze_congruency:')
print(pd.crosstab(df['GazeColorConsistency'], df['gaze_congruency'], margins=True))

# initiation_time check: should be non-null for all rows
print(f'\ninitiation_time — missing: {df["initiation_time"].isna().sum()}')
print(df.groupby('action_consistency')['initiation_time'].describe()[['count', 'mean', 'min', 'max']].round(1))

out = f'{base}\\05_gaze_congruency.xlsx'
df.to_excel(out, index=False)
print(f'\nSaved: {out}')
print(f'Shape: {df.shape}')

Loaded: 7913 rows × 23 cols

gaze_congruency:
gaze_congruency
incongruent    3966
congruent      3947
Name: count, dtype: int64

action_consistency:
action_consistency
inconsistent    2669
consistent      2645
control         2599
Name: count, dtype: int64

E-Prime GazeColorConsistency vs calculated gaze_congruency:
gaze_congruency       congruent  incongruent   All
GazeColorConsistency                              
consinstent                1314            0  1314
inconsistent                  0         1285  1285
offline                    2633         2681  5314
All                        3947         3966  7913

initiation_time — missing: 0
                     count   mean    min     max
action_consistency                              
consistent          2645.0  638.3   26.0  2734.0
control             2599.0  449.7  350.0   550.0
inconsistent        2669.0  654.6   24.0  2621.0

Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data

## Step 06 — RT outlier trimming (Python ±2SD)

For each `Subject × gaze_congruency × action_consistency` cell, drop trials more than 2 SD from
that cell's own mean, then take the mean of what remains.

**This replaces the manual SPSS step.** SPSS's `2w.SPS` script has a confirmed bug (see
`SPSS_2w_script_bug_summary.txt`): its last line, `compute FiltRT = filt*rt.`, relies on
multiplication to propagate the exclusion flag — but SPSS treats `SYSMIS * 0` as `0`, not `SYSMIS`.
Any trial with a genuine RT of exactly 0 (a missing-response code, not a real reaction time) that
SPSS's own filter correctly flags for exclusion ends up silently kept as a literal zero instead.
On this dataset that meant SPSS under-trimmed by 37 trials (7913→7587 instead of the correct
7913→7550). Python trimming is now canonical for all three experiments — no SPSS round-trip, no
numeric w1/w2 recoding needed (the previous "export to SPSS format" step is gone; grouping is done
directly on the string labels `gaze_congruency`/`action_consistency`).

In [15]:
df = pd.read_excel(f'{base}\\05_gaze_congruency.xlsx')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')

grp = df.groupby(['Subject', 'gaze_congruency', 'action_consistency'])['rt']
stats = grp.agg(['mean', 'std']).reset_index()
df = df.merge(stats, on=['Subject', 'gaze_congruency', 'action_consistency'])
keep_mask = (df['rt'] >= df['mean'] - 2*df['std']) & (df['rt'] <= df['mean'] + 2*df['std'])
n_before = len(df)
trimmed = df[keep_mask].copy()
n_after = len(trimmed)
print(f'Trials before trim: {n_before}')
print(f'Trials after +/-2SD trim: {n_after}  (removed {n_before-n_after}, {(n_before-n_after)/n_before*100:.2f}%)')

cellmeans = trimmed.groupby(['Subject', 'gaze_congruency', 'action_consistency'])['rt'].mean().reset_index()
cellmeans.columns = ['sub', 'gaze_congruency', 'action_consistency', 'rt_mean']
print(f'Cell means shape: {cellmeans.shape}')

out = f'{base}\\06_trimmed.xlsx'
cellmeans.to_excel(out, index=False)
print(f'Saved: {out}')

Loaded: 7913 rows × 29 cols
Trials before trim: 7913
Trials after +/-2SD trim: 7550  (removed 363, 4.59%)
Cell means shape: (294, 4)
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\06_trimmed.xlsx


## Step 07 — Participant-level mean RT outlier removal

Excludes participants whose overall mean (mean of their 6 cell means) exceeds grand mean + 2 SD.

Note: with Python trimming now canonical, the excluded subject changes from the earlier SPSS-based
run (ss67, which was only 0.9ms over its cutoff) to **ss17** here — individual trials shifting
between the two trimming methods moves each subject's mean RT slightly, which can flip who crosses
a margin this thin. Final N (48) happens to be unchanged, but the excluded individual is not the
same person — any results keyed to "who was excluded" should be re-checked, not assumed identical.

In [16]:
df = pd.read_excel(f'{base}\\06_trimmed.xlsx')
subj_means = df.groupby('sub')['rt_mean'].mean().reset_index()
subj_means.columns = ['sub', 'overall_mean_rt']
group_mean = subj_means['overall_mean_rt'].mean()
group_sd = subj_means['overall_mean_rt'].std()
cutoff = group_mean + 2*group_sd
excluded = subj_means[subj_means['overall_mean_rt'] > cutoff]
exclude_ids = excluded['sub'].tolist()
print(f'Grand mean RT : {group_mean:.1f} ms')
print(f'Grand SD      : {group_sd:.1f} ms')
print(f'Cutoff        : {cutoff:.1f} ms (mean + 2 SD)')
print(f'Excluded ({len(excluded)} subject(s)):')
for _, row in excluded.iterrows():
    print(f'  ss {int(row["sub"])}: mean RT = {row["overall_mean_rt"]:.1f} ms (+{row["overall_mean_rt"]-cutoff:.1f} ms over cutoff)')
clean = df[~df['sub'].isin(exclude_ids)].copy()
print(f'Remaining: {clean["sub"].nunique()} subjects, {len(clean)} rows')
out = f'{base}\\07_no_outliers.xlsx'
clean.to_excel(out, index=False)
print(f'Saved: {out}')

Grand mean RT : 519.2 ms
Grand SD      : 95.2 ms
Cutoff        : 709.6 ms (mean + 2 SD)
Excluded (1 subject(s)):
  ss 17: mean RT = 714.9 ms (+5.3 ms over cutoff)
Remaining: 48 subjects, 288 rows
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\07_no_outliers.xlsx


## Step 08 — Reshape to long + wide

Data is already one row per subject × condition after Step 06/07, and already string-labeled
(no numeric w1/w2 to decode anymore) — this step is now just a pivot to wide format.

In [17]:
df = pd.read_excel(f'{base}\\07_no_outliers.xlsx')
long = df[['sub', 'gaze_congruency', 'action_consistency', 'rt_mean']].copy()
n_subs = long['sub'].nunique()
print(f'Long format: {len(long)} rows ({n_subs} subjects x 6 conditions = {n_subs*6})')

col_map = {
    ('incongruent', 'inconsistent'): 'incong_incons', ('incongruent', 'consistent'): 'incong_cons', ('incongruent', 'control'): 'incong_ctrl',
    ('congruent', 'inconsistent'): 'cong_incons', ('congruent', 'consistent'): 'cong_cons', ('congruent', 'control'): 'cong_ctrl',
}
wide = df.pivot(index='sub', columns=['gaze_congruency', 'action_consistency'], values='rt_mean')
wide.columns = [col_map[c] for c in wide.columns]
wide = wide.reset_index()
print(f'Wide format: {wide.shape[0]} subjects x {wide.shape[1]} columns')

out = f'{base}\\08_ranova_input.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    long.to_excel(writer, sheet_name='long', index=False)
    wide.to_excel(writer, sheet_name='wide', index=False)
print(f'Saved: {out}  (sheets: long, wide)')

Long format: 288 rows (48 subjects x 6 conditions = 288)
Wide format: 48 subjects x 7 columns
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\08_ranova_input.xlsx  (sheets: long, wide)


## Step 09 — Mean initiation time per subject (ANCOVA covariate)

For the planned ANCOVA (see thesis discussion on the `initiation_time` column),
the covariate must be a single, comparable value per subject — and it should only be computed from
trials where it means what the theory says it means. `initiation_time` is `SlideXeroA.RT` on action
trials (a real behavioral measure — how long the participant took to initiate their response) but
`XeroDur` on control trials (an experimenter-imposed random delay, unrelated to anything the
participant did). Averaging across both would mix a genuine measure with arbitrary noise, and there
is no equivalent "prediction" construct in control trials for the covariate to represent.

So this computes the mean **only across action trials** (consistent + inconsistent, congruency
doesn't matter for this average) — control trials are excluded from this calculation entirely, and
any ANCOVA using this covariate should itself be restricted to the action-condition cells.

In [18]:
df = pd.read_excel(f'{base}\\05_gaze_congruency.xlsx')
action_only = df[df['action_consistency'].isin(['consistent', 'inconsistent'])]
init_time = action_only.groupby('Subject')['initiation_time'].mean().reset_index()
init_time.columns = ['sub', 'mean_initiation_time']
print(f'Computed for {len(init_time)} subjects (from {len(action_only)} action trials)')
print(init_time.describe().to_string())
out = f'{base}\\09_initiation_time.xlsx'
init_time.to_excel(out, index=False)
print(f'Saved: {out}')

Computed for 49 subjects (from 5314 action trials)
            sub  mean_initiation_time
count  49.00000             49.000000
mean   35.55102            647.558423
std    21.37177            243.967545
min     3.00000            264.357798
25%    17.00000            491.821429
50%    30.00000            619.895238
75%    55.00000            750.081818
max    70.00000           1427.468468
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\09_initiation_time.xlsx


## Step 10 — Merge question ratings + initiation time into wide RT data

Left join on `sub` — keeps only the 48 subjects present in the final exp data.

In [19]:
rt_wide = pd.read_excel(f'{base}\\08_ranova_input.xlsx', sheet_name='wide')
q_wide = pd.read_excel(os.path.join(os.path.dirname(base), 'questions', '04_aggregated.xlsx'), sheet_name='wide')
init_time = pd.read_excel(f'{base}\\09_initiation_time.xlsx')

merged = rt_wide.merge(q_wide, on='sub', how='left').merge(init_time, on='sub', how='left')
print(f'Subjects in exp data:       {rt_wide["sub"].nunique()}')
print(f'Subjects in questions data: {q_wide["sub"].nunique()}')
print(f'Subjects in merged output:  {merged["sub"].nunique()}')
dropped = set(q_wide['sub']) - set(rt_wide['sub'])
if dropped:
    print(f'Dropped from questions (not in exp data): ss {sorted(dropped)}')
missing_init = merged[merged['mean_initiation_time'].isna()]
if len(missing_init):
    print(f'WARNING: missing initiation time for: {missing_init["sub"].tolist()}')
print(f'Columns ({merged.shape[1]}): {list(merged.columns)}')

out = f'{base}\\10_questions_data_added.xlsx'
merged.to_excel(out, index=False)
print(f'Saved: {out}')

Subjects in exp data:       48
Subjects in questions data: 53
Subjects in merged output:  48
Dropped from questions (not in exp data): ss [1, 2, 17, 45, 59]
Columns (20): ['sub', 'cong_cons', 'cong_ctrl', 'cong_incons', 'incong_cons', 'incong_ctrl', 'incong_incons', 'consistent_caused', 'consistent_control', 'consistent_predict', 'control_caused', 'control_control', 'control_predict', 'inconsistent_caused', 'inconsistent_control', 'inconsistent_predict', 'Age', 'Sex', 'Handedness', 'mean_initiation_time']
Saved: C:\Users\Kaloyan Zdravkov\Desktop\Congitive Science\MasterThesis\GC + agency\experiment2\data_processing\exp_data\10_questions_data_added.xlsx


## Step 11 — Export to JASP

After opening in JASP: set `sub`, `Sex`, `Handedness` to Nominal. For the RT rANOVA, use the 6
`*_cons`/`*_incons`/`*_ctrl` columns. For the ANCOVA, use only the 4 action-condition columns
(`cong_cons`, `cong_incons`, `incong_cons`, `incong_incons`) with `mean_initiation_time` as
covariate — do not include the `*_ctrl` columns in that analysis (see Step 09 note).

**This re-exports the analysis file — any existing `.jasp` file built on the old SPSS-trimmed data
is now stale** (different excluded subject, different trial-level RT values) and needs its
analyses re-run against this new `analysis.csv`.

Also saves `11_merged_initTime_for_ANCOVA.xlsx` in `exp_data/` — a stable xlsx copy of the same merged data (same content as `10_questions_data_added.xlsx`), following the numbered-checkpoint naming convention, meant for continued JASP or Python analysis work without changing if the pipeline is re-run.

In [ ]:
jasp_dir = os.path.join(base, 'jasp')
os.makedirs(jasp_dir, exist_ok=True)
csv_path = os.path.join(jasp_dir, 'analysis.csv')
merged.to_csv(csv_path, index=False, encoding='utf-8')
print(f'Saved: {csv_path}')

working_path = f'{base}\\11_merged_initTime_for_ANCOVA.xlsx'
merged.to_excel(working_path, index=False)
print(f'Saved: {working_path}  (stable xlsx copy for continued JASP/Python analysis work)')
print(f'Subjects: {merged.shape[0]}  |  Columns: {merged.shape[1]}')